In [ ]:
import ast
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
# import seaborn as sns

# Load palettes
RAW_CSV = r"C:\Users\nina\Documents\UNI\SEM\thesis\coding\game_color_analysis\data\new_game_info.csv"
# RAW_CSV = r"C:\Users\nina\Documents\UNI\SEM\thesis\coding\game_color_analysis\data\game_info.csv"
# RAW_CSV = r"C:\Users\nina\Documents\UNI\SEM\thesis\coding\game_color_analysis\data\color_palettes.csv"

# PROCESSED_CSV = "processed_color_data.csv"
df = pd.read_csv(RAW_CSV)
# df.info()

# strings to python lists
df['Palette'] = df['Palette'].apply(ast.literal_eval)
# df["Palette"] = df["Palette"].apply(lambda x: ast.literal_eval(x) if x != "[]" else []) #pd.notnull(x) or != [] ?
# df["Genres"] = df["Genres"].apply(lambda x: ast.literal_eval(x) if x != "[]" else [])
# df["Themes"] = df["Themes"].apply(lambda x: ast.literal_eval(x) if x != "[]" else [])

# all colors from a game in 1 row
palette_by_game = df.explode('Palette').groupby(['Year', 'Game'])['Palette']
# palette_by_genre = df.explode('Palette').explode('Genres').groupby(['Genres'])
# palette_by_theme = df.explode('Palette').explode('Themes').groupby(['Themes'])

# colors_by_game = palette_by_game.agg(set).reset_index()
colors_by_game = palette_by_game.agg(Counter).reset_index()
# colors_by_game.iloc[0]['Palette'].most_common(1)
def get_top_colors(x, n=3):
    return Counter(x).most_common(n)
top_colors_by_game = palette_by_game.agg(lambda x: get_top_colors(x, 3)).reset_index()

# Explode a následné spočítání výskytů každé barvy pro každou hru
col_counts = (df.explode('Palette')
                 .groupby(['Year', 'Game', 'Palette'])
                 .size()
                 .reset_index(name='Count'))

# Seřazení od nejčastějších barev
col_counts = col_counts.sort_values(['Year', 'Game', 'Count'], ascending=[True, True, False])

top_5_per_year = (col_counts
                  .sort_values(['Year', 'Count'], ascending=[True, False])
                  .groupby('Year')
                  .head(5))
# 1. Spočítáme celkový výskyt barev v každé dekádě
top_5_per_decade = (df.explode('Palette')
                    .groupby(['Decade', 'Palette'])
                    .size()
                    .reset_index(name='Count'))

# 2. Seřadíme a vybereme 5 nejčastějších pro každou dekádu
top_5_per_decade = (top_5_per_decade
                    .sort_values(['Decade', 'Count'], ascending=[True, False])
                    .groupby('Decade')
                    .head(5))


# color_by_genre = palette_by_genre['Palette'].agg(set).reset_index()
# color_by_theme = palette_by_theme['Palette'].agg(set).reset_index()

# h = colors_by_game.loc[colors_by_game["Year"] == 1975]
# f = Counter(h["Palette"].explode()).most_common(3)
# g = (palette_by_game.groupby("Year")["Palette"].value_counts())#.groupby(level=0).head(5))

# Dominant colors per year (top 5)
# dominant_colors_year = {
#     year: [c for c, _ in Counter(colors).most_common(5)]
#     for year, colors in year_colors.items()
# }

